# IEEE-CIS - Fraud Pattern / Scenario Analysis

Behavioural feature engineering and rule-based fraud scenarios on top of the IEEE-CIS data. Loads `preprocessed.pkl` from `preprocessing.ipynb` instead of re-deriving it, to avoid duplicating that pipeline.

Model training lives in `IEEE-CIS-SET1.ipynb` / `IEEE-CIS-SET2.ipynb`, not here.

In [ ]:
import pandas as pd
import pickle

# load the cleaned/split IEEE-CIS data produced by preprocessing.ipynb
with open("preprocessed.pkl", "rb") as f:
    data = pickle.load(f)

df_clean = data["df_clean"]
df_train = data["df_train"]
df_test = data["df_test"]

print("Loaded df_clean:", df_clean.shape)


In [ ]:
# fraud rate by card network
card4_fraud = df_clean.groupby('card4')['isFraud'].agg(['sum', 'count'])
card4_fraud['fraud_rate'] = (card4_fraud['sum'] / card4_fraud['count'] * 100).round(2)
print("Fraud rate by card network:")
print(card4_fraud.sort_values('fraud_rate', ascending=False))

card6_fraud = df_clean.groupby('card6')['isFraud'].agg(['sum', 'count'])
card6_fraud['fraud_rate'] = (card6_fraud['sum'] / card6_fraud['count'] * 100).round(2)
print("\nFraud rate by card type:")
print(card6_fraud.sort_values('fraud_rate', ascending=False))

In [ ]:
# fraud rate by product code
product_fraud = df_clean.groupby('ProductCD')['isFraud'].agg(['sum','count'])
product_fraud['fraud_rate'] = (product_fraud['sum']/product_fraud['count']*100).round(2)
print("Fraud rate by product code:")
print(product_fraud.sort_values('fraud_rate', ascending=False))

## Behavioural analysis

In [ ]:
# reload unscaled data, sorted by card + time, for behavioural features
import pandas as pd
import numpy as np

df_original = pd.read_parquet("merged_data.parquet")

required_cols = ['card1', 'TransactionDT']
missing = [c for c in required_cols if c not in df_original.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df_original = df_original.sort_values(
    ['card1', 'TransactionDT']
).reset_index(drop=True)

print("Loaded original data:", df_original.shape)
print("Columns available:", df_original.columns.tolist()[:10], "...")

print("\nVerification (should NOT look like z-scores, i.e. not centered near 0):")
print(df_original[['TransactionDT', 'TransactionAmt', 'card1']].describe().loc[['mean', 'std', 'min', 'max']])

In [ ]:
# engineer velocity/behavioural features (time gap, amount jump, address & product changes)
df_original['days_since_last_tx'] = (
    df_original.groupby('card1')['TransactionDT']
    .diff().fillna(999) / 86400
)

df_original['prev_amount'] = (
    df_original.groupby('card1')['TransactionAmt']
    .shift(1).fillna(0)
)

df_original['prev_addr1'] = (
    df_original.groupby('card1')['addr1']
    .shift(1)
)

df_original['address_changed'] = (
    df_original['addr1'] != df_original['prev_addr1']
).astype(int)

df_original['prev_product'] = (
    df_original.groupby('card1')['ProductCD']
    .shift(1)
)

df_original['product_changed'] = (
    df_original['ProductCD'] != df_original['prev_product']
).astype(int)

card_mean = df_original.groupby('card1')['TransactionAmt'].transform('mean')
card_std = df_original.groupby('card1')['TransactionAmt'].transform('std').fillna(1)
df_original['amount_deviation'] = (
    (df_original['TransactionAmt'] - card_mean) / card_std
)

email_risk = df_original.groupby(
    'P_emaildomain')['isFraud'].transform('mean')
df_original['email_domain_risk'] = email_risk.fillna(
    df_original['isFraud'].mean())

df_original['amount_jump'] = (
    df_original['TransactionAmt'] / 
    df_original['prev_amount'].replace(0, 1)
)

print("Underlying features built")

In [ ]:
# scenario 1: dormant card reactivated with a high-value purchase
scenario1 = (
    (df_original['days_since_last_tx'] > 90) &
    (df_original['TransactionAmt'] > 200)
)

df_original['scenario1_dormant_reactivation'] = scenario1.astype(int)

total_s1 = scenario1.sum()
fraud_s1 = df_original[scenario1]['isFraud'].sum()
fraud_rate_s1 = df_original[scenario1]['isFraud'].mean() * 100

print("SCENARIO 1 — Dormant Card Sudden Reactivation:")
print(f"  Definition: Card inactive 90+ days, purchase >£200")
print(f"  Transactions flagged: {total_s1:,}")
print(f"  Actual fraud among flagged: {fraud_s1:,}")
print(f"  Fraud rate when flagged: {fraud_rate_s1:.2f}%")
print(f"  Baseline fraud rate: {df_original['isFraud'].mean()*100:.2f}%")
print(f"  Risk multiplier: {fraud_rate_s1/df_original['isFraud'].mean()/100:.1f}x higher than baseline")
print()

In [ ]:
# scenario 2: address changed just before/after a transaction
scenario2 = (
    (df_original['address_changed'] == 1) &
    (df_original['days_since_last_tx'] < 1) &
    (df_original['days_since_last_tx'] > 0)
)

df_original['scenario2_rapid_location_change'] = scenario2.astype(int)

total_s2 = scenario2.sum()
fraud_s2 = df_original[scenario2]['isFraud'].sum()
fraud_rate_s2 = df_original[scenario2]['isFraud'].mean() * 100

print("SCENARIO 2 — Rapid Location Change (Card Cloning Signal):")
print(f"  Definition: Address changed, same card within 24 hours")
print(f"  Transactions flagged: {total_s2:,}")
print(f"  Actual fraud among flagged: {fraud_s2:,}")
print(f"  Fraud rate when flagged: {fraud_rate_s2:.2f}%")
print(f"  Baseline fraud rate: {df_original['isFraud'].mean()*100:.2f}%")
print(f"  Risk multiplier: {fraud_rate_s2/df_original['isFraud'].mean()/100:.1f}x higher than baseline")
print()

In [ ]:
# scenario 3: card testing (small purchase, then a big one, same day)
scenario3 = (
    (df_original['amount_jump'] > 10) &
    (df_original['days_since_last_tx'] < 1) &
    (df_original['days_since_last_tx'] > 0) &
    (df_original['prev_amount'] < 20)
)

df_original['scenario3_card_testing'] = scenario3.astype(int)

total_s3 = scenario3.sum()
fraud_s3 = df_original[scenario3]['isFraud'].sum()
fraud_rate_s3 = df_original[scenario3]['isFraud'].mean() * 100

print("SCENARIO 3 — Card Testing Pattern:")
print(f"  Definition: Amount jumped 10x+ within 24h after small purchase")
print(f"  Transactions flagged: {total_s3:,}")
print(f"  Actual fraud among flagged: {fraud_s3:,}")
print(f"  Fraud rate when flagged: {fraud_rate_s3:.2f}%")
print(f"  Baseline fraud rate: {df_original['isFraud'].mean()*100:.2f}%")
print(f"  Risk multiplier: {fraud_rate_s3/df_original['isFraud'].mean()/100:.1f}x higher than baseline")
print()

In [ ]:
# scenario 4: product category anomaly (possible account takeover)
scenario4 = (
    (df_original['product_changed'] == 1) &
    (df_original['amount_deviation'] > 2)
)

df_original['scenario4_product_anomaly'] = scenario4.astype(int)

total_s4 = scenario4.sum()
fraud_s4 = df_original[scenario4]['isFraud'].sum()
fraud_rate_s4 = df_original[scenario4]['isFraud'].mean() * 100

print("SCENARIO 4 — Product Category Anomaly (Account Takeover):")
print(f"  Definition: New product type + amount 2+ std above card average")
print(f"  Transactions flagged: {total_s4:,}")
print(f"  Actual fraud among flagged: {fraud_s4:,}")
print(f"  Fraud rate when flagged: {fraud_rate_s4:.2f}%")
print(f"  Baseline fraud rate: {df_original['isFraud'].mean()*100:.2f}%")
print(f"  Risk multiplier: {fraud_rate_s4/df_original['isFraud'].mean()/100:.1f}x higher than baseline")
print()

In [ ]:
# scenario 5: risky email domain + new address
baseline_fraud_rate = df_original['isFraud'].mean()

scenario5 = (
    (df_original['email_domain_risk'] > baseline_fraud_rate * 2) &
    (df_original['address_changed'] == 1)
)

df_original['scenario5_risky_email_new_address'] = scenario5.astype(int)

total_s5 = scenario5.sum()
fraud_s5 = df_original[scenario5]['isFraud'].sum()
fraud_rate_s5 = df_original[scenario5]['isFraud'].mean() * 100

print("SCENARIO 5 — High Risk Email + New Address:")
print(f"  Definition: Email domain fraud rate 2x baseline + address changed")
print(f"  Transactions flagged: {total_s5:,}")
print(f"  Actual fraud among flagged: {fraud_s5:,}")
print(f"  Fraud rate when flagged: {fraud_rate_s5:.2f}%")
print(f"  Baseline fraud rate: {df_original['isFraud'].mean()*100:.2f}%")
print(f"  Risk multiplier: {fraud_rate_s5/df_original['isFraud'].mean()/100:.1f}x higher than baseline")
print()

In [ ]:
# combine all 5 scenarios into one risk score
df_original['fraud_risk_score'] = (
    df_original['scenario1_dormant_reactivation'] +
    df_original['scenario2_rapid_location_change'] +
    df_original['scenario3_card_testing'] +
    df_original['scenario4_product_anomaly'] +
    df_original['scenario5_risky_email_new_address']
)

print("COMBINED RISK SCORE ANALYSIS:")
print("(0 = no scenarios triggered, 5 = all scenarios triggered)")
print()

risk_analysis = df_original.groupby('fraud_risk_score')['isFraud'].agg(
    ['sum', 'count', 'mean'])
risk_analysis.columns = ['fraud_count', 'total_transactions', 'fraud_rate']
risk_analysis['fraud_rate'] = (risk_analysis['fraud_rate'] * 100).round(2)

print(risk_analysis)

print()
print("KEY FINDING:")
print(f"  Score 0 (no flags): {risk_analysis.loc[0,'fraud_rate']}% fraud rate")
if 1 in risk_analysis.index:
    print(f"  Score 1 (1 flag):   {risk_analysis.loc[1,'fraud_rate']}% fraud rate")
if 2 in risk_analysis.index:
    print(f"  Score 2 (2 flags):  {risk_analysis.loc[2,'fraud_rate']}% fraud rate")
if 3 in risk_analysis.index:
    print(f"  Score 3+ (3+ flags):{risk_analysis.loc[3:,'fraud_rate'].max()}% fraud rate")

In [ ]:
# translate findings into practical fraud-detection recommendations
print("=" * 60)
print("PRACTICAL FRAUD DETECTION RECOMMENDATIONS")
print("=" * 60)
print()
print("Based on scenario analysis of 590,540 IEEE-CIS transactions:")
print()

print(f"SCENARIO 1 (Dormant Reactivation):")
print(f"  {total_s1:,} transactions flagged")
print(f"  Fraud rate {fraud_rate_s1:.1f}% vs baseline {baseline_fraud_rate*100:.1f}%")
print(f"  RECOMMENDATION: Trigger SMS/email verification")
print(f"  when card inactive 90+ days makes purchase >£200")
print()

print(f"SCENARIO 2 (Rapid Location Change):")
print(f"  {total_s2:,} transactions flagged")
print(f"  Fraud rate {fraud_rate_s2:.1f}% vs baseline {baseline_fraud_rate*100:.1f}%")
print(f"  RECOMMENDATION: Block transaction pending")
print(f"  customer confirmation when billing address")
print(f"  changes within 24 hours of last transaction")
print()

print(f"SCENARIO 3 (Card Testing):")
print(f"  {total_s3:,} transactions flagged")
print(f"  Fraud rate {fraud_rate_s3:.1f}% vs baseline {baseline_fraud_rate*100:.1f}%")
print(f"  RECOMMENDATION: Flag account for review when")
print(f"  amount jumps 10x+ within 24 hours of small purchase")
print()

print(f"SCENARIO 4 (Product Anomaly):")
print(f"  {total_s4:,} transactions flagged")
print(f"  Fraud rate {fraud_rate_s4:.1f}% vs baseline {baseline_fraud_rate*100:.1f}%")
print(f"  RECOMMENDATION: Request additional authentication")
print(f"  when purchase category differs from card history")
print(f"  combined with unusually high amount")
print()

print(f"SCENARIO 5 (Risky Email + New Address):")
print(f"  {total_s5:,} transactions flagged")
print(f"  Fraud rate {fraud_rate_s5:.1f}% vs baseline {baseline_fraud_rate*100:.1f}%")
print(f"  RECOMMENDATION: Add to manual review queue")
print(f"  when high-risk email domain is combined with")
print(f"  a newly seen shipping address")
print()

print("COMBINED SCORE RECOMMENDATION:")
print("  Score 0: Process normally")
print("  Score 1: Log for monitoring")
print("  Score 2: Request 2FA verification")
print("  Score 3+: Block pending manual review")
print()
print("=" * 60)